# Section 1: Exploratory Data Analysis (EDA) and Pre-Processing




## Section 1.1: EDA-Dataset Overview


In [ ]:
# ======================================
# Core Libraries
# ======================================

# Numerical computing (arrays, math operations)
import numpy as np

# Data manipulation and analysis
import pandas as pd

# Mathematical utilities
import math

# ======================================
# Data Visualisation
# ======================================

# Matplotlib for basic plotting
import matplotlib.pyplot as plt

# Seaborn for statistical visualisations
import seaborn as sns

# ======================================
# Preprocessing & Feature Engineering
# ======================================

# Scale numerical features (mean=0, std=1)
from sklearn.preprocessing import StandardScaler

# Scale numerical features to range [0,1]
from sklearn.preprocessing import MinMaxScaler

# Handle missing values (mean/median/mode)
from sklearn.impute import SimpleImputer

# Apply different preprocessing to column groups
from sklearn.compose import ColumnTransformer

# Encode categorical variables
from sklearn.preprocessing import OneHotEncoder

# Split dataset into train and test sets
from sklearn.model_selection import train_test_split

# ======================================
# Clustering Algorithms
# ======================================

# K-Means clustering
from sklearn.cluster import KMeans

# BIRCH clustering (efficient for large datasets)
from sklearn.cluster import Birch

# Clustering evaluation metric
from sklearn.metrics import silhouette_score, adjusted_rand_score

# ======================================
# Regression Models & Metrics
# ======================================

# Linear regression model
from sklearn.linear_model import LinearRegression

# Random Forest regressor (ensemble model)
from sklearn.ensemble import RandomForestRegressor

# Regression evaluation metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Cross-validation utilities for regression
from sklearn.model_selection import KFold, cross_validate

# ======================================
# Classification Models & Metrics
# ======================================

# Logistic Regression classifier
from sklearn.linear_model import LogisticRegression

# Random Forest classifier
from sklearn.ensemble import RandomForestClassifier

# Classification evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix
)

# Cross-validation and hyperparameter tuning
from sklearn.model_selection import StratifiedKFold, GridSearchCV

# ======================================
# Other Utilities
# ======================================

# Suppress warnings for cleaner output
import warnings
warnings.simplefilter('ignore')

# ======================================
# Notebook Display Settings
# ======================================

# Display plots inline (Jupyter)
%matplotlib inline

# Adjust DataFrame display
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 120)

# Set consistent seaborn style
sns.set(style='whitegrid')


# Section 1.1.a Load the dataset

In [ ]:
# Define dataset path
path = "bank-full-updated.csv"

# Load dataset into DataFrame
df = pd.read_csv(path)

# Display dataset shape
display(df.shape)

# Preview first 5 rows
display(df.head())

# Inspect data types
display(df.info())

display(df.describe().T)

The summary presents the average, minimum, and maximum values for each numerical feature. Variables such as balance and duration exhibit extremely high maximum values, indicating potential outliers. The target variable y has a low mean, confirming that the dataset is strongly imbalanced.

# Section 1.1.b Finding missing values

In [ ]:
# Count missing values in each column
missing_per_col = df.isna().sum()

# Sort columns by missing values (highest first) and show top 10
missing_per_col_sorted = missing_per_col.sort_values(ascending=False)
display(missing_per_col_sorted.head(10))

# Plot missing values per column
plt.figure(figsize=(10, 3))  # Set figure size
plt.bar(missing_per_col.index, missing_per_col.values)  # Bar chart
plt.title("Missing values per column")
plt.ylabel("Count of missing values")
plt.xlabel("Columns")
plt.xticks(rotation=90)  # Rotate labels for readability
plt.show()

# Count missing values per row
missing_per_row = df.isna().sum(axis=1)

plt.figure(figsize=(10,3))
plt.hist(missing_per_row.values,bins = range(0,df.shape[1]))
plt.title('Missing values per row')
plt.ylabel('Count of rows with corresponding number of missing values')
plt.ylabel('Number of missing values')
plt.xticks(rotation=90)
plt.show()

The above analysis shows that there are no missing values in any column of the dataset. Both the column wise and row wise checks confirm that all entries are complete.


# Section 1.1.c Checking unique values

In [ ]:
# Select categorical columns (exclude numerical columns)
cat_cols = df.select_dtypes(exclude='number').columns

# Loop through each categorical column
for col in cat_cols:
    print(f"\nColumn: {col}")  # Print column name
    print(f"Unique values ({df[col].nunique()}):")  # Print number of unique values
    print(df[col].unique())  # Display unique categories

# Section 1.1.d: Unknown Value Analysis (Row-wise & Column-wise)

In [ ]:
# Row-wise unknown count
unknown_per_row = (df[cat_cols]
                   .astype(str)
                   .applymap(lambda x: x.lower() == "unknown")
                  ).sum(axis=1)

# Sort and display top 5 rows with most unknown values
unknown_per_row_sorted = unknown_per_row.sort_values(ascending=False)
display(unknown_per_row_sorted.head(5))

# Plot histogram (row-wise)
plt.figure(figsize=(10,3))
plt.hist(unknown_per_row.values, bins=range(0, df.shape[1]))
plt.title('Histogram of unknown values per row')
plt.xlabel('Number of unknown values')
plt.ylabel('Number of rows')
plt.show()

# ---- Column-wise unknown count ----
unknown_per_col = (df[cat_cols]
                   .astype(str)
                   .applymap(lambda x: x.lower() == "unknown")
                  ).sum()

# Sort columns by highest unknown count
unknown_per_col = unknown_per_col.sort_values(ascending=False)
display(unknown_per_col.head())

# Plot bar chart (column-wise)
plt.figure(figsize=(10,5))
plt.bar(unknown_per_col.index, unknown_per_col.values)
plt.title('Unknown values per column')
plt.xlabel('Categorical Columns')
plt.ylabel('Count of unknown values')
plt.xticks(rotation=45)
plt.show()



The analysis shows the presence of unknown values in the dataset. Most rows contain between one and three unknown entries, while very few contain four. At the column level, four features include unknown values, with poutcome having the highest count, indicating placeholder missing data.

In [ ]:
# Select numerical columns and count unique values in each
num_cols = df.select_dtypes(include='number').columns
df[num_cols].nunique()


In [ ]:
df['day'].unique()

In [ ]:
df['pdays'].unique()[:10]

In [ ]:
# Count negative values in each numerical column
display((df[num_cols] < 0).sum().sort_values(ascending=False))

# Count total duplicate rows in the dataset
n_dups = df.duplicated().sum()
display(n_dups)


## Section 1.1.e Target variable analysis (`y`)

In [ ]:
df["y"].unique()

In [ ]:
# Calculate class distribution (counts and percentages)
target_counts = df["y"].value_counts(dropna=False)
target_perc = df["y"].value_counts(normalize=True, dropna=False) * 100

# Display counts and percentages together
display(pd.DataFrame({
    "count": target_counts,
    "percent": target_perc.round(2)
}))

# Plot target variable distribution
plt.figure(figsize=(5, 3))
sns.countplot(x="y", data=df)
plt.title("Target distribution (y)")
plt.xlabel("y")
plt.ylabel("Count")
plt.show()



The target variable y is not balanced. Around 88.3% pf customers did not subscribe, while only 11.7% subscribed.

# Section 1.1.f Distribution of Numerical Variables


In [ ]:
# Select numerical columns
num_cols = df.select_dtypes(include='number').columns

# Plot histograms for all numerical features
df[num_cols].hist(figsize=(12,8))

# Display the plots
plt.show()


The histograms show that most numerical variables are not normally distributed. Age is mainly concentrated between 30 and 50 years. Balance, duration, campaign, pdays, and previous are strongly right-skewed with many near-zero values and some large outliers. The target variable y is clearly imbalanced. These patterns suggest scaling and outlier treatment during preprocessing.

# Section 1.1.g Categorical Feature Distributions

In [ ]:
# Select categorical columns
cat_cols = df.select_dtypes(include='object').columns

# Set overall figure size
plt.figure(figsize=(14, 10))

# Loop through categorical columns and plot countplots
for i, col in enumerate(cat_cols, 1):
    # Create subplots (3 plots per row)
    plt.subplot(math.ceil(len(cat_cols) / 3), 3, i)
    
    # Plot category frequency
    sns.countplot(data=df, x=col)
    
    plt.xticks(rotation=45)  # Rotate labels for readability
    plt.title(col)

plt.tight_layout()  # Adjust spacing
plt.show()

The categorical distributions indicate dominance of certain categories across multiple features. In job, management, blue-collar, and technician are most frequent. Most customers are married, and secondary education is the most common level. Very few customers have credit default = yes. More customers hold housing loans than personal loans, and most contacts occur via cellular. The poutcome feature contains many unknown values, indicating missing prior campaign information. Overall, several categorical variables are imbalanced and require careful preprocessing.

# Section 1.1.h Categorical Variables vs Target (Subscription Rate)

In [ ]:
# Helper: find the "positive" label in y 
# (works if y is {0,1} or {'no','yes'})
y_unique = list(pd.Series(df['y'].unique()).dropna())
pos_label = 1 if 1 in y_unique else ('yes' if 'yes' in y_unique else y_unique[-1])

def subscription_rate_table(col):
    """Return a table of subscription rates by category."""
    # Create cross-tab normalized by row (percentage within each category)
    tab = pd.crosstab(df[col], df['y'], normalize='index')
    
    # Sort by positive class subscription rate (if exists)
    if pos_label in tab.columns:
        tab = tab.sort_values(by=pos_label, ascending=False)
        
    return tab
def plot_subscription_rate(col, top_n=20):
    """Bar plot of subscription rate (positive class) by category."""
    
    # Get subscription rate table
    tab = subscription_rate_table(col)
    
    # Check if positive label exists
    if pos_label not in tab.columns:
        print(f"Positive label {pos_label} not found in y columns for {col}.")
        display(tab.head(top_n))
        return
    
    # Extract positive class rates
    rates = tab[pos_label]
    
    # Limit to top N categories (if too many)
    if len(rates) > top_n:
        rates = rates.head(top_n)
    
    # Plot subscription rate
    plt.figure(figsize=(10,4))
    rates.plot(kind='bar')
    plt.ylabel('Subscription rate')
    plt.title(f'Subscription rate by {col} (top {min(top_n, len(tab))})')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    return tab

In [ ]:
# Subscription rate by job

# Check if 'job' column exists in dataset
if 'job' in df.columns:
    
    # Plot subscription rate for each job category
    job_rate = plot_subscription_rate('job', top_n=20)
    
    # Display top 10 categories with highest subscription rate
    display(job_rate.head(10))
    
else:
    print("Column 'job' not found in dataset.")


Categorical distributions show dominance of certain groups. Management, blue-collar, and technician jobs are most frequent. Most customers are married, and secondary education dominates. Few have credit default = yes. Housing loans exceed personal loans, contacts are mainly cellular, and poutcome contains many unknown values, indicating imbalance and required preprocessing.

In [ ]:
# Subscription rate by education
if 'education' in df.columns:
    edu_rate = plot_subscription_rate('education', top_n=20)
    display(edu_rate)
else:
    print("Column 'education' not found in dataset.")

Subscription rates vary by education level. Customers with tertiary education have the highest rate (15%), followed by the unknown category. Secondary education shows a moderate rate, while primary education has the lowest (8.6%). Overall, higher education is associated with a greater likelihood of subscribing, making education an important predictive feature.

In [ ]:
# Check selected columns for "unknown" impact on target variable
unknown_cols = [c for c in ['job','education','contact','poutcome'] if c in df.columns]

for col in unknown_cols:
    
    # Check if column contains "unknown"
    if (df[col] == 'unknown').any():
        
        # Create temporary boolean flag (True if value is 'unknown')
        flag = (df[col] == 'unknown')
        
        # Calculate subscription rate split by unknown vs not unknown
        tab_unknown = pd.crosstab(flag, df['y'], normalize='index')
        print(f"\nSubscription rate split by {col}_is_unknown:")
        display(tab_unknown)

        # Plot distribution of target by unknown flag
        plt.figure(figsize=(5,3))
        sns.countplot(x=flag, hue='y', data=df)
        plt.title(f"'{col}' unknown vs target")
        plt.xlabel(f"{col}_is_unknown")
        plt.tight_layout()
        plt.show()
        
    else:
        print(f"\nNo 'unknown' values found in '{col}'.")


# Section 1.1.i Outlier

In [ ]:
# Dictionary to store outlier information for each numerical column
outlier_summary = {}

# Loop through each numerical column
for col in num_cols:
    
    # Calculate first and third quartiles
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    
    # Calculate Interquartile Range (IQR)
    IQR = Q3 - Q1

    # Define lower and upper bounds using IQR rule
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Identify outliers (values outside bounds)
    outlier_mask = (df[col] < lower_bound) | (df[col] > upper_bound)

    # Store summary statistics
    outlier_summary[col] = {
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": int(outlier_mask.sum())
    }

# Convert dictionary to DataFrame for easy viewing
outlier_df = pd.DataFrame(outlier_summary).T

# Display outlier summary table
outlier_df


In [ ]:
# Boxplots for numerical columns (to visually detect outliers)

plt.figure(figsize=(12, 9))  # Set overall figure size

plot_cols = 4  # Number of plots per row
plot_rows = int(math.ceil(len(num_cols) / plot_cols))  # Calculate required rows

# Loop through numerical columns and create boxplots
for i, col in enumerate(num_cols):
    plt.subplot(plot_rows, plot_cols, i + 1)
    sns.boxplot(x=df[col])  # Boxplot for each numerical feature
    plt.title(col)

plt.tight_layout()  # Adjust spacing between plots
plt.show()


The boxplots confirm the presence of outliers across several numerical features. Balance, duration, campaign, pdays, and previous show many extreme values beyond the whiskers. Age has only a few higher-end outliers, with most values between 30 and 50, while day appears relatively evenly distributed. ID behaves like an index and adds little value. Overall, significant skewness and outliers suggest the need for capping, transformation, or scaling during preprocessing.

# Section 1.1.j Correlation Analysis

In [ ]:
# Compute correlation matrix for numerical features
corr = df[num_cols].corr()

# Plot heatmap of correlations
plt.figure(figsize=(10,6))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.show()


The heatmap shows generally weak correlations among numerical features, indicating low multicollinearity. Duration has the strongest positive relationship with the target, meaning longer calls increase subscription likelihood. A moderate correlation between pdays and previous is expected. Overall, the low correlations are favourable for modelling.

# Section 1.2: PreProcessing

In [ ]:
# ===============================
# Feature Separation Before Preprocessing
# ===============================

# Store ID separately (not useful for modelling)
ID = df['ID']

# Define target variable for classification
y = df['y']

# Store duration separately (to avoid data leakage in classification)
duration = df['duration']

# Create feature matrix excluding ID, target (y), and duration
# Safe for clustering and as input features for models
df_PrePross = df.drop(columns=['ID', 'y', 'duration'])

# Display remaining feature columns
df_PrePross.columns


# ===============================
# Separate Numerical and Categorical Features
# ===============================

# Identify numerical columns
num_cols = df_PrePross.select_dtypes(include='number').columns.tolist()

# Identify categorical columns
cat_cols = df_PrePross.select_dtypes(include='object').columns.tolist()

# Print grouped columns
print("Numeric:", num_cols)
print("Categorical:", cat_cols)


# Section 1.2.a Handling "unknown" (before imputation/encoding)

In [ ]:
# Create a copy of the dataset for preprocessing
df_base = df_PrePross.copy()

# Replace 'unknown' values in categorical columns with NaN
# (So they can be treated as missing values)
df_base[cat_cols] = df_base[cat_cols].replace('unknown', np.nan)

# Count missing values per column
missing_per_col = df_base.isna().sum()

# Sort columns by number of missing values (highest first)
missing_per_col_sorted = missing_per_col.sort_values(ascending=False)

# Display top 5 columns with most missing values
display(missing_per_col_sorted.head(5))


Several categorical features contained "unknown", representing missing information rather than valid categories. These values were converted to NaN to ensure proper missing-value handling. This enables systematic imputation and prevents models from misinterpreting "unknown" as meaningful data during preprocessing.


# Section 1.2.b Imputation

In [ ]:
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

df_base[num_cols] = num_imputer.fit_transform(df_base[num_cols])
df_base[cat_cols] = cat_imputer.fit_transform(df_base[cat_cols])
check_missing_val = df_base.isna().sum().sort_values(ascending=False)
display(check_missing_val.head(5))


Missing values were imputed rather than removed to avoid unnecessary loss of data and potential sampling bias. Numerical features were imputed using the median to reduce the influence of outliers, while categorical features were imputed using the most frequent category to maintain realistic category distributions. This approach ensures data completeness while preserving the underlying structure of the dataset.

# Section 1.2.c Capping

In [ ]:
def cap_iqr(data, col):
    """
    Cap outliers in a numerical column using the IQR rule.
    Any value below/above the IQR bounds is clipped to the boundary.
    """
    # Compute quartiles and IQR
    q1 = data[col].quantile(0.25)
    q3 = data[col].quantile(0.75)
    iqr = q3 - q1

    # Define IQR-based lower/upper limits
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    # Clip values outside the bounds (winsorization)
    data[col] = data[col].clip(lower, upper)

    return lower, upper


# Columns where extreme values can distort scaling/clustering/model training
cols_to_cap = ['balance', 'campaign', 'previous']

# Record min/max BEFORE capping (for comparison)
before_stats = (
    df_base[cols_to_cap]
    .agg(['min', 'max'])
    .T
    .rename(columns={'min': 'min_before', 'max': 'max_before'})
)

# Create a capped version of the dataset (keep original df_base unchanged)
df_capped = df_base.copy()

# Store the calculated cap boundaries for each column
cap_bounds = {}

# Apply IQR capping column-by-column
for col in cols_to_cap:
    lower, upper = cap_iqr(df_capped, col)
    cap_bounds[col] = (lower, upper)

# Record min/max AFTER capping (to verify effect)
after_stats = (
    df_capped[cols_to_cap]
    .agg(['min', 'max'])
    .T
    .rename(columns={'min': 'min_after', 'max': 'max_after'})
)

# Combine before/after stats into one table
cap_comparison = before_stats.join(after_stats)
display(cap_comparison)

# Sanity checks: after capping, max should not increase and min should not decrease
for col in cols_to_cap:
    assert cap_comparison.loc[col, 'max_after'] <= cap_comparison.loc[col, 'max_before'], \
        f"Max not reduced for {col}"
    assert cap_comparison.loc[col, 'min_after'] >= cap_comparison.loc[col, 'min_before'], \
        f"Min not increased for {col}"


Outliers in selected numerical features were capped using the IQR method to reduce the influence of extreme values without removing observations. This preserves dataset size while improving model stability, scaling behaviour, and clustering performance during subsequent modelling stages.


# Section 1.2.d Splitting the data into train and test

In [ ]:
# Features (pre-call only) — remove poutcome
cols_to_remove = ['poutcome']

X = df_capped.drop(columns=cols_to_remove)

# Target (classification)
y = df['y']

# Stratify keeps the class ratio similar in train and test (important for bank marketing)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Section 1.2.e Approach 1: StandardScaler + One-Hot Encoding

In [ ]:
# 1) Identify numeric and categorical columns (based on training data)
num_cols = X_train.select_dtypes(include='number').columns.tolist()
cat_cols = X_train.select_dtypes(include='object').columns.tolist()

# 2) Sanity checks
assert len(num_cols) + len(cat_cols) == X_train.shape[1], " Column split mismatch"
assert X_train.isna().sum().sum() == 0, " X_train still has missing values"
assert X_test.isna().sum().sum() == 0, " X_test still has missing values"

# 3) Build preprocessing transformer (keep ALL dummy columns)
preprocess_approach_1 = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop=None), cat_cols)
    ],
    remainder='drop'
)

# 4) Fit ONLY on training data, transform both
X1_train = preprocess_approach_1.fit_transform(X_train)
X1_test  = preprocess_approach_1.transform(X_test)

feature_names_1 = preprocess_approach_1.get_feature_names_out()

print("X1_train:", X1_train.shape)
print("X1_test :", X1_test.shape)

# Sanity checks
assert X1_train.shape[1] == len(feature_names_1), " Feature name count mismatch"
assert X1_test.shape[1] == X1_train.shape[1], " Train/Test feature dimension mismatch"
assert np.isfinite(X1_train.toarray()[:5]).all() if hasattr(X1_train, "toarray") else np.isfinite(np.asarray(X1_train)[:5]).all(), "❌ NaN/inf in X1_train"
assert np.isfinite(X1_test.toarray()[:5]).all() if hasattr(X1_test, "toarray") else np.isfinite(np.asarray(X1_test)[:5]).all(), "❌ NaN/inf in X1_test"

# 5) Preview columns safely (convert only first N rows of TRAIN)
N = 5
X1_preview = X1_train[:N].toarray() if hasattr(X1_train, "toarray") else np.asarray(X1_train[:N])

X1_train_preview_df = pd.DataFrame(X1_preview, columns=feature_names_1)
display(X1_train_preview_df.head())

# Section 1.2.f Approach 2: MinMax Scaling + Frequency Encoding

In [ ]:
# 1) Identify numeric and categorical columns (based on training data)
num_cols = X_train.select_dtypes(include='number').columns.tolist()
cat_cols = X_train.select_dtypes(include='object').columns.tolist()


# 2) Sanity checks
assert len(num_cols) + len(cat_cols) == X_train.shape[1], "Column split mismatch"
assert X_train.isna().sum().sum() == 0, "X_train still has missing values"
assert X_test.isna().sum().sum() == 0, "X_test still has missing values"

# 3) Frequency encoding maps learned from TRAIN only
freq_maps = {}
X2_train_df = X_train.copy()
X2_test_df = X_test.copy()

for col in cat_cols:
    freq_map = X_train[col].value_counts(normalize=True)   # fit on TRAIN only
    freq_maps[col] = freq_map

    X2_train_df[col] = X2_train_df[col].map(freq_map)
    X2_test_df[col] = X2_test_df[col].map(freq_map).fillna(0.0)  # unseen categories -> 0

# Sanity checks: categorical columns now numeric and within [0,1]
for col in cat_cols:
    assert pd.api.types.is_numeric_dtype(X2_train_df[col]), f"{col} not numeric after freq encoding (train)"
    assert pd.api.types.is_numeric_dtype(X2_test_df[col]), f"{col} not numeric after freq encoding (test)"
    assert X2_train_df[col].min() >= 0 and X2_train_df[col].max() <= 1, f"{col} out of [0,1] (train)"
    assert X2_test_df[col].min() >= 0 and X2_test_df[col].max() <= 1, f" {col} out of [0,1] (test)"

# 4) MinMax scaling fit on TRAIN only (numerical columns)
scaler_app2 = MinMaxScaler()
X2_train_df[num_cols] = scaler_app2.fit_transform(X2_train_df[num_cols])
X2_test_df[num_cols] = scaler_app2.transform(X2_test_df[num_cols])

# Sanity checks: numeric columns in ~[0,1]
for col in num_cols:
    assert X2_train_df[col].min() >= -1e-9 and X2_train_df[col].max() <= 1 + 1e-9, f"{col} out of [0,1] (train)"
    assert X2_test_df[col].min() >= -1e-6 and X2_test_df[col].max() <= 1 + 1e-6, f"{col} out of [0,1] (test)"

# 5) Final matrices
X2_train = X2_train_df.values
X2_test  = X2_test_df.values

print("X2_train:", X2_train.shape)
print("X2_test :", X2_test.shape)

# Final sanity checks
assert X2_train.shape[1] == X2_test.shape[1], "Train/Test feature dimension mismatch"
assert np.isfinite(X2_train).all(), "NaN/inf found in X2_train"
assert np.isfinite(X2_test).all(), "NaN/inf found in X2_test"

# Preview
display(X2_train_df.head())



The second preprocessing approach was selected due to its suitability for distance-based learning tasks and its ability to produce a compact, fully numeric representation of the data. In this approach, categorical variables are encoded using frequency encoding, while numerical variables are scaled using Min–Max normalization. One-hot encoding is commonly used to convert categorical variables into a machine-readable format while avoiding the introduction of ordinal relationships between nominal categories (James et al., 2021).

Frequency encoding replaces each category with its empirical frequency in the training data, allowing categorical variables to be represented on a continuous scale without introducing artificial ordering or substantially increasing dimensionality. Compared to one-hot encoding, this results in a significantly lower-dimensional feature space, which is advantageous for algorithms that rely on distance calculations.

Min–Max scaling ensures that all numerical features are bounded within the [0,1] range. Feature scaling is important because distance-based and gradient-based algorithms are sensitive to feature magnitude, and unscaled variables can disproportionately influence model behaviour (Han et al., 2011). This prevents features with large natural scales, such as account balance or number of contacts, from dominating the distance metric and allows categorical and numerical features to contribute more evenly to the learning process.

Together, these transformations produce a compact and well-scaled feature representation that preserves relative similarity between observations. This makes the second preprocessing approach particularly well suited to clustering and other distance-based models, where interpretability of distances and stability of cluster assignments are critical.


# Section 2: Clustering

## Section 2.1: KMean

In [ ]:
X_km_train = X2_train  

wcss = []
sil_scores = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_km_train)

    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(X_km_train, labels))

# Elbow plot
plt.figure(figsize=(7,4))
plt.plot(list(K_range), wcss, marker='o')
plt.title("Elbow Method (KMeans on TRAIN)")
plt.xlabel("k")
plt.ylabel("WCSS / Inertia")
plt.show()

# Silhouette plot
plt.figure(figsize=(7,4))
plt.plot(list(K_range), sil_scores, marker='o')
plt.title("Silhouette Score (KMeans on TRAIN)")
plt.xlabel("k")
plt.ylabel("Silhouette Score")
plt.show()

# Print scores (helps for write-up)
for k, s in zip(K_range, sil_scores):
    print(f"k={k}: silhouette={s:.4f}")


After removing post-call variables such as poutcome, K-Means clustering was evaluated using the Elbow Method and Silhouette Score. K-Means was selected as a baseline segmentation method due to its efficiency and suitability for partitioning numerical data into compact, spherical clusters (MacQueen, 1967). The silhouette coefficient was used to evaluate cluster quality as it measures both intra-cluster cohesion and inter-cluster separation (Rousseeuw, 1987). 

The Silhouette Score peaked at k = 4 (0.2569), with the Elbow plot also flattening beyond this point. Therefore, four clusters were selected, indicating moderate but meaningful separation consistent with overlapping financial behaviours in real-world banking data. Internal clustering metrics are appropriate when ground-truth labels are unavailable, as is typical in unsupervised customer segmentation tasks (Tan et al., 2019).


In [ ]:
best_k = 4

kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
kmeans_final.fit(X2_train)

train_clusters = kmeans_final.labels_
test_clusters  = kmeans_final.predict(X2_test)

# Add cluster feature
X2_train_df["cluster"] = train_clusters
X2_test_df["cluster"]  = test_clusters

print("Train cluster counts:")
print(X2_train_df["cluster"].value_counts().sort_index())

print("\nTest cluster counts:")
print(X2_test_df["cluster"].value_counts().sort_index())


# Cluster Distribution Analysis

In [ ]:
X_train_with_cluster = X_train.copy()
X_train_with_cluster["cluster"] = train_clusters

# Median summary for numeric columns
cluster_numeric_summary = X_train_with_cluster.groupby("cluster")[num_cols].median()
display(cluster_numeric_summary)


Clustering identified four customer segments driven mainly by balance and campaign frequency, revealing meaningful, actionable differences in marketing engagement.

In [ ]:
for col in cat_cols:
    display(
        pd.crosstab(X_train_with_cluster["cluster"],
                    X_train_with_cluster[col],
                    normalize="index").round(3)
    )


Cluster profiling reveals clear segmentation primarily driven by financial and campaign-related variables rather than basic demographics such as age.

Cluster 0 – General Customer Base (60%) represents mainstream customers with moderate balances, no prior contact history, and no personal loans, forming the largest homogeneous group.

Cluster 1 – Repeated Contact Segment (19%) shows the highest campaign frequency, indicating customers requiring multiple marketing attempts.

Cluster 2 – High Debt Segment (13%) is characterised by the lowest balances and a high proportion of personal loans, suggesting financially constrained customers.

Cluster 3 – High-Value Previously Engaged Segment (8%) contains high-balance customers with positive prior marketing engagement.

# Section 2.2: BIRCH

In [ ]:

sil_scores_birch = []
K_range = range(2, 11)

for k in K_range:
    birch = Birch(n_clusters=k)  # you can tune threshold later if needed
    labels = birch.fit_predict(X2_train)
    sil_scores_birch.append(silhouette_score(X2_train, labels))

plt.figure(figsize=(7,4))
plt.plot(list(K_range), sil_scores_birch, marker='o')
plt.title("Silhouette Score (BIRCH - TRAIN)")
plt.xlabel("k")
plt.ylabel("Silhouette Score")
plt.show()

for k, s in zip(K_range, sil_scores_birch):
    print(f"k={k}: silhouette={s:.4f}")


# Section 2.3: Model Selection: KMeans vs BIRCH (Internal Metrics + Stability)

In [ ]:

# Evaluate both algorithms for the same k-range
k_range = range(2, 11)
rows = []

for k in k_range:
    # ---- KMeans ----
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_labels = km.fit_predict(X2_train)

    rows.append({
        "model": "KMeans",
        "k": k,
        "silhouette": silhouette_score(X2_train, km_labels),
        "davies_bouldin": davies_bouldin_score(X2_train, km_labels),
        "calinski_harabasz": calinski_harabasz_score(X2_train, km_labels),
        "inertia": km.inertia_
    })

    # ---- BIRCH ----
    br = Birch(n_clusters=k)
    br_labels = br.fit_predict(X2_train)

    rows.append({
        "model": "BIRCH",
        "k": k,
        "silhouette": silhouette_score(X2_train, br_labels),
        "davies_bouldin": davies_bouldin_score(X2_train, br_labels),
        "calinski_harabasz": calinski_harabasz_score(X2_train, br_labels),
        "inertia": np.nan  # inertia is not defined for BIRCH
    })

results_df = pd.DataFrame(rows)

# Full comparison table
display(results_df.sort_values(["model", "k"]))

# Best k per model (based on highest silhouette)
best_by_model = results_df.loc[results_df.groupby("model")["silhouette"].idxmax()].reset_index(drop=True)
display(best_by_model)


In [ ]:

def stability_same_data(model_name, X, k, n_runs=10):
    labels_list = []

    for seed in range(n_runs):
        if model_name == "KMeans":
            model = KMeans(n_clusters=k, random_state=seed, n_init=10)
        else:
            model = Birch(n_clusters=k)

        labels = model.fit_predict(X)
        labels_list.append(labels)

    # pairwise ARI
    scores = []
    for i in range(len(labels_list)):
        for j in range(i+1, len(labels_list)):
            scores.append(adjusted_rand_score(labels_list[i], labels_list[j]))

    return np.mean(scores)


# Compute stability for each model at its best k (from silhouette)
stability_rows = []

for _, r in best_by_model.iterrows():
    stability_rows.append({
        "model": r["model"],
        "k": int(r["k"]),
        "silhouette": float(r["silhouette"]),
        "stability_ARI": stability_same_data(
            r["model"], X2_train, int(r["k"])
        )
    })

stability_df = pd.DataFrame(stability_rows)
display(stability_df)



### Final Model Selection

Both K-Means and BIRCH were evaluated using Silhouette Score and
stability analysis.

BIRCH (k = 2) achieved the highest Silhouette Score and perfect
stability, indicating very compact clusters. However, it produced
only two clusters, resulting in a coarse segmentation that provides
limited marketing insight.

K-Means (k = 4) produced slightly lower separation but maintained
good stability (ARI ≈ 0.72) while generating more granular and
actionable customer segments.

Given the marketing objective of deriving meaningful customer
profiles, K-Means with k = 4 was selected as the final clustering
model.


# Section 3: Regression 

In [ ]:
# Regression target (duration)

y_reg_train = df.loc[X_train.index, "duration"]
y_reg_test  = df.loc[X_test.index, "duration"]

y_reg_train_log = np.log1p(y_reg_train)
y_reg_test_log  = np.log1p(y_reg_test) 

print("y_reg_train shape:", y_reg_train_log.shape)
print("y_reg_test shape:", y_reg_test_log.shape)



# Section 3.1: Linear Regression 

In [ ]:
X2_train_df.head()

In [ ]:
lr = LinearRegression()
lr.fit(X2_train_df, y_reg_train)

y_pred_lr = lr.predict(X2_test_df)

rmse_lr = np.sqrt(mean_squared_error(y_reg_test, y_pred_lr))
r2_lr = r2_score(y_reg_test, y_pred_lr)

print("Linear Regression Results:")
print("RMSE:", round(rmse_lr, 2))
print("R2 Score:", round(r2_lr, 4))
print(f"Pred range (test): [{y_pred_lr.min():.2f}, {y_pred_lr.max():.2f}]")
print(f"True range (test): [{y_reg_test.min():.2f}, {y_reg_test.max():.2f}]")


# Section 3.2: Randon Forest Regressor

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X2_train_df, y_reg_train)

y_pred_rf = rf.predict(X2_test_df)

rmse_rf = np.sqrt(mean_squared_error(y_reg_test, y_pred_rf))
r2_rf = r2_score(y_reg_test, y_pred_rf)

print("\nRandom Forest Results:")
print("RMSE:", round(rmse_rf, 2))
print("R2 Score:", round(r2_rf, 4))


# Section 3.3: Regression Model Selection (CV)

In [ ]:


# ---------- Use your prepared data ----------
X_train_reg = X2_train_df.copy()
X_test_reg  = X2_test_df.copy()

y_train_log = y_reg_train_log
y_test_log  = y_reg_test_log

# ---------- Cross-validation setup ----------
cv = KFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2"
}

models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )
}

# ---------- CV comparison ----------
rows = []
for name, model in models.items():
    scores = cross_validate(model, X_train_reg, y_train_log, cv=cv, scoring=scoring)
    
    rows.append({
        "model": name,
        "CV_RMSE_mean": -scores["test_rmse"].mean(),
        "CV_RMSE_std":  scores["test_rmse"].std(),
        "CV_MAE_mean":  -scores["test_mae"].mean(),
        "CV_R2_mean":    scores["test_r2"].mean()
    })

reg_cv_results = pd.DataFrame(rows).sort_values("CV_RMSE_mean")
display(reg_cv_results)

# ---------- Select best model ----------
best_model_name = reg_cv_results.iloc[0]["model"]
best_model = models[best_model_name]

print(f"\nSelected model based on CV: {best_model_name}")



### Regression Model Selection

Random Forest regression was considered because ensemble tree methods can capture complex non-linear relationships and interactions without strong parametric assumptions (Breiman, 2001). Linear regression was used as a baseline model due to its interpretability and well-understood statistical properties (James et al., 2021).

Both Linear Regression and Random Forest were evaluated using cross-validation to ensure robust and generalisable performance. Based on the CV results, Random Forest achieved a lower mean RMSE (0.8929) compared to Linear Regression (0.9063), indicating slightly better predictive accuracy. It also produced a lower MAE (0.6928 vs 0.6981), suggesting improved average error performance.

Importantly, Random Forest achieved a higher mean R² score (0.0634) than Linear Regression (0.0353), demonstrating a better ability to explain variance in the target variable, although the overall explanatory power remains modest. The relatively small standard deviation of RMSE (0.0036) further indicates that the Random Forest model is stable across folds.

The improved performance of Random Forest is expected because it can capture non-linear relationships and feature interactions that Linear Regression cannot model effectively. However, the low R² values across both models suggest that the available features have limited predictive strength for the regression target.

Overall, Random Forest was selected as the final regression model due to its consistently better cross-validated performance and greater modelling flexibility.


# Section 4: Classification

In [ ]:
# Train final regression model
rf_final = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_final.fit(X2_train_df, y_reg_train)

# Predict duration
pred_duration_train = rf_final.predict(X2_train_df)
pred_duration_test  = rf_final.predict(X2_test_df)


In [ ]:
X2_train_df["predicted_duration"] = pred_duration_train
X2_test_df["predicted_duration"]  = pred_duration_test


# Section 4.1: Logistic Regression

In [ ]:
log_clf = LogisticRegression(max_iter=1000, class_weight='balanced')


log_clf.fit(X2_train_df, y_train)

y_pred_log = log_clf.predict(X2_test_df)
y_prob_log = log_clf.predict_proba(X2_test_df)[:,1]

print("Logistic Regression Results")
print("Accuracy:", accuracy_score(y_test, y_pred_log))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_log))
print(classification_report(y_test, y_pred_log))


# Section 4.2: Random Forest Classifier

In [ ]:
rf_clf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_clf.fit(X2_train_df, y_train)

y_pred_rf = rf_clf.predict(X2_test_df)
y_prob_rf = rf_clf.predict_proba(X2_test_df)[:,1]

print("Random Forest Results")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))
print(classification_report(y_test, y_pred_rf))


# Section 4.3: Classification Model selection

In [ ]:
def eval_classifier(name, y_true, y_pred, y_prob):
    print(f"\n{name}")
    print("Accuracy :", accuracy_score(y_true, y_pred))
    print("ROC-AUC   :", roc_auc_score(y_true, y_prob))
    print("PR-AUC    :", average_precision_score(y_true, y_prob))
    print("Precision :", precision_score(y_true, y_pred))
    print("Recall    :", recall_score(y_true, y_pred))
    print("F1        :", f1_score(y_true, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

eval_classifier("Logistic Regression", y_test, y_pred_log, y_prob_log)
eval_classifier("Random Forest", y_test, y_pred_rf, y_prob_rf)


Because the subscription outcome is imbalanced, evaluation metrics beyond accuracy (such as ROC-AUC and Precision-Recall) are more informative for assessing classifier performance. Class weighting was also applied to mitigate bias toward the majority class, a common issue in imbalanced classification problems (He and Garcia, 2009).

Logistic Regression showed reasonable detection ability (ROC-AUC = 0.71) and high recall (0.68) but produced many false positives due to very low precision. Random Forest performed better overall, achieving higher ROC-AUC (0.78), PR-AUC (0.39), and accuracy (0.89), although its recall was lower. Because Random Forest provided stronger overall performance and more reliable predictions, it was selected as the final classification model.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models_and_params = {
    "LogReg": (
        LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear"),
        {
            "C": [0.01, 0.1, 1, 10],
            "penalty": ["l1", "l2"]
        }
    ),
    "RF": (
        RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced_subsample"),
        {
            "n_estimators": [200, 400],
            "max_depth": [None, 10, 20],
            "min_samples_split": [2, 10],
            "min_samples_leaf": [1, 5]
        }
    )
}


In [ ]:
best_models = {}

for name, (model, params) in models_and_params.items():
    gs = GridSearchCV(
        estimator=model,
        param_grid=params,
        scoring="average_precision",   # or "roc_auc"
        cv=cv,
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X2_train_df, y_train)
    best_models[name] = gs

    print(f"\n{name} best CV score:", gs.best_score_)
    print("Best params:", gs.best_params_)


Hyperparameter tuning improved both models, but Random Forest achieved a higher cross-validated score (0.388) than Logistic Regression (0.268). The best Logistic Regression used L1 regularisation (C = 10), while Random Forest performed best with 200 trees and controlled leaf size (min_samples_leaf = 5). Because Random Forest consistently showed stronger cross-validated performance and better ability to capture complex patterns, it was selected as the final tuned classification model.


In [ ]:
for name, gs in best_models.items():
    best_clf = gs.best_estimator_
    y_pred = best_clf.predict(X2_test_df)
    y_prob = best_clf.predict_proba(X2_test_df)[:, 1]

    print(f"\n===== {name} (Tuned) Test Performance =====")
    print("ROC-AUC:", roc_auc_score(y_test, y_prob))
    print("PR-AUC :", average_precision_score(y_test, y_prob))
    print(classification_report(y_test, y_pred))


After hyperparameter tuning, Logistic Regression showed moderate performance with ROC-AUC of 0.71 and high recall for the positive class (0.68), meaning it detected many potential subscribers. However, its very low precision (0.20) resulted in many false positives and lower overall accuracy (0.64).  

In comparison, the tuned Random Forest clearly performed better. It achieved higher ROC-AUC (0.79), PR-AUC (0.40), and overall accuracy (0.87), indicating stronger class separation and improved predictive power. Importantly, Random Forest provided a more balanced precision–recall trade-off for the minority class (precision = 0.45, recall = 0.49), making its positive predictions more reliable.  

Therefore, Random Forest was selected as the final classification model because it delivers better overall performance and more practical value for marketing decision-making.


# References

Breiman, L., 2001. Random forests. Machine Learning, 45(1), pp.5–32.

Han, J., Kamber, M. and Pei, J., 2011. Data Mining: Concepts and Techniques. 3rd ed. Morgan Kaufmann.

He, H. and Garcia, E.A., 2009. Learning from imbalanced data. IEEE Transactions on Knowledge and Data Engineering, 21(9), pp.1263–1284.

James, G., Witten, D., Hastie, T. and Tibshirani, R., 2021. An Introduction to Statistical Learning. 2nd ed. Springer.

MacQueen, J., 1967. Some methods for classification and analysis of multivariate observations. In: Proceedings of the Fifth Berkeley Symposium on Mathematical Statistics and Probability.

Rousseeuw, P.J., 1987. Silhouettes: a graphical aid to the interpretation and validation of cluster analysis. Journal of Computational and Applied Mathematics, 20, pp.53–65.

Tan, P.N., Steinbach, M., Karpatne, A. and Kumar, V., 2019. Introduction to Data Mining. 2nd ed. Pearson.
